# ロボットアーム2台のトランプ対戦（PyBullet / Google Colab）
- ♥ロボットA と ♠ロボットB が対座し、「一斉のせ！」で手札からランダムに1枚ずつ出します。
- 強さ: **A(=1)が最強**、それ以外は数が大きい方が勝ち。同じ数はドロー（両者カードを手持ちに戻してやり直し）。
- 勝った側が相手のカードを**自分の陣地**へ運び、自分の出したカードは手持ちに戻します。
- 終了条件: ドローを除いて **13回決着したら終了**（13は奇数なので同点なし）。陣地の枚数が多い方が勝者。
- Colabで「ランタイム → すべてのセルを実行」でそのまま動きます。

In [ ]:
# ===== 準備: ライブラリと日本語フォントのインストール（Colab用）=====
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pybullet", "imageio", "imageio-ffmpeg"], check=True)
if "google.colab" in sys.modules:   # Colabでは字幕用の日本語フォントを入れる
    subprocess.run("apt-get -qq install -y fonts-noto-cjk > /dev/null", shell=True)

In [ ]:
# ===== 設定（CONFIG）: ここの数値を変えるだけで挙動を調整できます =====
CONFIG = {
    "seed": 2026,               # 乱数シード（同じ値なら毎回同じ試合を再現）
    "decisive_rounds": 13,      # 決着ラウンド数（ドローを除く）。奇数なので同点にならない
    "max_draws": 30,            # ドローの上限（無限ループ防止の安全装置）
    "base_distance": 0.70,      # 各アームの土台と卓の中心の距離 [m]
    "play_y": 0.15,             # 場に出すカード位置（中心からの距離）[m]
    "hand_xy": (-0.30, 0.40),   # 手札置き場 (x, 中心からの距離) [m]
    "camp_xy": (0.32, 0.40),    # 陣地（取ったカード置き場）(x, 中心からの距離) [m]
    "hover_z": 0.25,            # カードを運ぶときの高さ [m]
    "grip_z": 0.06,             # カードをつかむ・置くときの手先の高さ [m]
    "card_size": (0.08, 0.12, 0.004),  # カードの大きさ 幅×奥行×厚さ [m]
    "steps_move": 8,            # 移動1区間あたりのフレーム数（大きいほどゆっくり）
    "steps_updown": 4,          # 上げ下げ1回あたりのフレーム数
    "fps": 20,                  # 動画のフレームレート
    "width": 480, "height": 360,  # 動画の解像度（軽め）
    "hold_frames": 10,          # 掛け声や勝敗表示で静止するフレーム数
    "final_hold_frames": 60,    # 最終結果を表示し続けるフレーム数（3秒）
    "out_mp4": "game.mp4",      # 出力ファイル名
}
CONFIG

In [ ]:
# ===== ゲームロジック（ロボットとは独立。ここだけでルールを検証できる）=====
import random

RANK_LABEL = {1: "A", 11: "J", 12: "Q", 13: "K"}
def label(suit, n):
    return f"{suit}{RANK_LABEL.get(n, str(n))}"

def strength(n):
    """カードの強さ。A(=1)は最強なので14として扱う。2〜13はそのまま。"""
    return 14 if n == 1 else n

def judge(a, b):
    """'A' / 'B' / 'draw' を返す。同じ数（A同士を含む）はドロー。"""
    if strength(a) == strength(b):
        return "draw"
    return "A" if strength(a) > strength(b) else "B"

def play_game(cfg):
    rng = random.Random(cfg["seed"])
    hand = {"A": list(range(1, 14)), "B": list(range(1, 14))}   # A=♥, B=♠
    camp = {"A": [], "B": []}                                   # 陣地（取ったカード）
    log, draws, decided = [], 0, 0
    while decided < cfg["decisive_rounds"] and hand["A"] and hand["B"]:
        a = rng.choice(hand["A"]); b = rng.choice(hand["B"])     # 一斉のせ！で同時にランダム1枚
        res = judge(a, b)
        if res == "draw":
            draws += 1                                          # 出したカードは手持ちに戻すだけ
            if draws > cfg["max_draws"]:
                break
        else:
            decided += 1
            loser = "B" if res == "A" else "A"
            taken = b if res == "A" else a
            hand[loser].remove(taken)                           # 負けた側のカードは相手の陣地へ
            camp[res].append((loser, taken))                    # 勝った側の自分のカードは手持ちに戻る
        log.append({"a": a, "b": b, "result": res, "decided": decided, "draws": draws,
                    "score": (len(camp["A"]), len(camp["B"]))})
    sa, sb = len(camp["A"]), len(camp["B"])
    winner = "A(♥)" if sa > sb else ("B(♠)" if sb > sa else "引き分け")
    return {"log": log, "camp": camp, "score": (sa, sb), "draws": draws, "winner": winner}

# ルールの単体テスト
assert judge(1, 13) == "A" and judge(13, 1) == "B"   # Aは最強
assert judge(1, 1) == "draw" and judge(7, 7) == "draw"
assert judge(10, 3) == "A" and judge(2, 9) == "B"
result = play_game(CONFIG)
for i, r in enumerate(result["log"], 1):
    print(f"{i:2d}: ♥{label('',r['a']):>2} vs ♠{label('',r['b']):>2} -> {r['result']:4s}  陣地 A={r['score'][0]} B={r['score'][1]}")
print("最終:", result["score"], "ドロー回数:", result["draws"], "勝者:", result["winner"])

In [ ]:
# ===== ロボット制御（PyBullet / ヘッドレス DIRECT モード + getCameraImage で録画）=====
import math, os, numpy as np, pybullet as p, pybullet_data, imageio
from PIL import Image, ImageDraw, ImageFont

def find_font(size):
    """日本語フォントを探す（Colabでは最初のセルで fonts-noto-cjk を入れる）"""
    for path in ["/usr/share/fonts/opentype/noto/NotoSansCJK-Bold.ttc",
                 "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",
                 "/usr/share/fonts/truetype/noto/NotoSansCJK-Regular.ttc"]:
        if os.path.exists(path):
            return ImageFont.truetype(path, size)
    return ImageFont.load_default()

class CardTable:
    def __init__(self, cfg):
        self.cfg = cfg
        if p.isConnected(): p.disconnect()
        p.connect(p.DIRECT)
        p.setAdditionalSearchPath(pybullet_data.getDataPath())
        p.loadURDF("plane.urdf")
        d = cfg["base_distance"]
        # A は y=-d から +y 向き、B は y=+d から -y 向きに対座させる
        self.arm = {"A": p.loadURDF("kuka_iiwa/model.urdf", [0, -d, 0], p.getQuaternionFromEuler([0, 0, math.pi/2]), useFixedBase=True),
                    "B": p.loadURDF("kuka_iiwa/model.urdf", [0, d, 0], p.getQuaternionFromEuler([0, 0, -math.pi/2]), useFixedBase=True)}
        self.side = {"A": -1, "B": 1}
        self.ee = 6
        self.down = p.getQuaternionFromEuler([0, math.pi, 0])  # 手先を真下に向ける
        self.q = {}
        for k, body in self.arm.items():
            self.q[k] = [0, 0.6, 0, -1.4, 0, 1.1, 0]
            self._set(k, self.q[k])
        # 卓のマット（見た目用、衝突なし）
        self._mat([0, 0, 0.001], [0.6, 0.35, 0.001], [0.1, 0.45, 0.2, 1])
        for k in "AB":
            s = self.side[k]
            self._mat([cfg["camp_xy"][0], s*cfg["camp_xy"][1], 0.002], [0.08, 0.1, 0.001], [0.95, 0.85, 0.2, 1])  # 陣地(黄)
            self._mat([cfg["hand_xy"][0], s*cfg["hand_xy"][1], 0.002], [0.08, 0.1, 0.001], [0.6, 0.6, 0.65, 1])   # 手札(灰)
        self.font_big, self.font = find_font(30), find_font(18)
        self.cards = {}        # (owner, n) -> body id （場に見えているカード）
        self.camp_count = {"A": 0, "B": 0}
        self.hand_count = {"A": 13, "B": 13}
        self.hand_bodies = {k: [self._card(k, self._hand_pos(k, i)) for i in range(13)] for k in "AB"}
        self.frames, self.caption = [], ("", "")

    def _mat(self, pos, half, rgba):
        v = p.createVisualShape(p.GEOM_BOX, halfExtents=half, rgbaColor=rgba)
        p.createMultiBody(0, -1, v, pos)

    def _card(self, owner, pos):
        w, h, t = self.cfg["card_size"]
        rgba = [0.85, 0.1, 0.1, 1] if owner == "A" else [0.1, 0.1, 0.15, 1]  # ♥=赤, ♠=黒
        v = p.createVisualShape(p.GEOM_BOX, halfExtents=[w/2, h/2, t/2], rgbaColor=rgba)
        return p.createMultiBody(0, -1, v, pos)

    def _hand_pos(self, k, i):
        x, y = self.cfg["hand_xy"]; return [x, self.side[k]*y, 0.004 + i*0.005]
    def _camp_pos(self, k, i):
        x, y = self.cfg["camp_xy"]; return [x, self.side[k]*y, 0.004 + i*0.005]

    def _set(self, k, q):
        for j, v in enumerate(q): p.resetJointState(self.arm[k], j, v)

    def _ik(self, k, pos):
        self._set(k, self.q[k])  # 現在姿勢から解かせて、なめらかな解を選ぶ
        sol = p.calculateInverseKinematics(self.arm[k], self.ee, pos, self.down, maxNumIterations=200)
        return list(sol[:7])

    def ee_pos(self, k):
        return p.getLinkState(self.arm[k], self.ee)[4]

    def move(self, targets, steps, carry=None):
        """targets: {arm: xyz}。複数アームを同時に関節空間で補間して動かす。
        carry: {arm: card body} 手先に追従させる（吸着グリッパの簡易表現）"""
        carry = carry or {}
        start = {k: list(self.q[k]) for k in targets}
        goal = {k: self._ik(k, pos) for k, pos in targets.items()}
        for s in range(1, steps + 1):
            t = 0.5 - 0.5*math.cos(math.pi*s/steps)   # 加減速のあるS字補間
            for k in targets:
                self.q[k] = [a + (b - a)*t for a, b in zip(start[k], goal[k])]
                self._set(k, self.q[k])
            for k, body in carry.items():
                x, y, z = self.ee_pos(k)
                p.resetBasePositionAndOrientation(body, [x, y, z - 0.045], [0, 0, 0, 1])
            self.snap()

    def pick_place(self, moves):
        """moves: {arm: (取る位置, 置く位置, 運ぶカード)} を同時に実行"""
        c, hz, gz = self.cfg, self.cfg["hover_z"], self.cfg["grip_z"]
        up = lambda xyz, z: [xyz[0], xyz[1], z]
        self.move({k: up(m[0], hz) for k, m in moves.items()}, c["steps_move"])
        self.move({k: up(m[0], gz + m[0][2]) for k, m in moves.items()}, c["steps_updown"])
        carry = {k: m[2] for k, m in moves.items()}
        self.move({k: up(m[0], hz) for k, m in moves.items()}, c["steps_updown"], carry)
        self.move({k: up(m[1], hz) for k, m in moves.items()}, c["steps_move"], carry)
        self.move({k: up(m[1], gz + m[1][2]) for k, m in moves.items()}, c["steps_updown"], carry)
        for k, m in moves.items():
            p.resetBasePositionAndOrientation(m[2], m[1], [0, 0, 0, 1])
        self.move({k: up(m[1], hz) for k, m in moves.items()}, c["steps_updown"])

    def snap(self, n=1):
        c = self.cfg
        view = p.computeViewMatrixFromYawPitchRoll([0, 0, 0.2], 2.0, 90, -35, 0, 2)
        proj = p.computeProjectionMatrixFOV(55, c["width"]/c["height"], 0.05, 10)
        img = p.getCameraImage(c["width"], c["height"], view, proj, renderer=p.ER_TINY_RENDERER)[2]
        im = Image.fromarray(np.reshape(img, (c["height"], c["width"], 4))[:, :, :3].astype(np.uint8))
        dr = ImageDraw.Draw(im)
        dr.rectangle([0, 0, c["width"], 34], fill=(0, 0, 0))
        dr.text((8, 5), self.caption[0], font=self.font, fill=(255, 255, 255))
        dr.text((10, 40), "♥ロボットA", font=self.font, fill=(200, 0, 0))
        dr.text((c["width"]-110, 40), "♠ロボットB", font=self.font, fill=(20, 20, 30))
        dr.rectangle([0, c["height"]-40, c["width"], c["height"]], fill=(0, 0, 0))
        dr.text((8, c["height"]-36), self.caption[1], font=self.font_big if len(self.caption[1]) < 18 else self.font, fill=(255, 230, 80))
        frame = np.array(im)
        for _ in range(n): self.frames.append(frame)

    def play_round(self, i, r):
        c, d = self.cfg, self.cfg["play_y"]
        sa, sb = r["score"]
        prev = (sa - (r["result"] == "A"), sb - (r["result"] == "B"))
        head = f"第{i}手  陣地 ♥A:{prev[0]}枚  ♠B:{prev[1]}枚  ドロー{r['draws'] - (r['result']=='draw')}回"
        self.caption = (head, "「一斉のせ！」"); self.snap(c["hold_frames"])
        # 1) 両アームが同時に手札の山から1枚取り、自分の正面に出す
        bodies, play_pos = {}, {}
        for k in "AB":
            self.hand_count[k] -= 1
            bodies[k] = self.hand_bodies[k].pop()
            play_pos[k] = [0, self.side[k]*d, 0.004]
        self.pick_place({k: (self._hand_pos(k, self.hand_count[k]), play_pos[k], bodies[k]) for k in "AB"})
        la, lb = label("♥", r["a"]), label("♠", r["b"])
        if r["result"] == "draw":
            self.caption = (head, f"{la} vs {lb} → ドロー！やり直し"); self.snap(c["hold_frames"])
            # 2) ドロー: 両者とも自分のカードを手持ちに戻す
            self.pick_place({k: (play_pos[k], self._hand_pos(k, self.hand_count[k]), bodies[k]) for k in "AB"})
            for k in "AB":
                self.hand_bodies[k].append(bodies[k]); self.hand_count[k] += 1
            return
        w = r["result"]; l = "B" if w == "A" else "A"
        self.caption = (head, f"{la} vs {lb} → {'♥A' if w=='A' else '♠B'}の勝ち！"); self.snap(c["hold_frames"])
        # 2) 勝者が相手のカードを自分の陣地へ運ぶ
        self.pick_place({w: (play_pos[l], self._camp_pos(w, self.camp_count[w]), bodies[l])})
        self.camp_count[w] += 1
        # 3) 勝者は自分の出したカードを手持ちに戻す（負けた側の手持ちは1枚減る）
        self.pick_place({w: (play_pos[w], self._hand_pos(w, self.hand_count[w]), bodies[w])})
        self.hand_bodies[w].append(bodies[w]); self.hand_count[w] += 1
        self.caption = (f"第{i}手  陣地 ♥A:{sa}枚  ♠B:{sb}枚  ドロー{r['draws']}回", f"{la} vs {lb} → {'♥A' if w=='A' else '♠B'}の勝ち！")
        self.snap(4)

    def save(self, path):
        imageio.mimsave(path, self.frames, fps=self.cfg["fps"], macro_block_size=8)

In [ ]:
# ===== メイン実行: ゲームを進めながらロボットを動かし、動画を書き出す =====
table = CardTable(CONFIG)
for i, r in enumerate(result["log"], 1):
    table.play_round(i, r)
sa, sb = result["score"]
table.caption = (f"試合終了  陣地 ♥A:{sa}枚  ♠B:{sb}枚  ドロー{result['draws']}回",
                 f"勝者 {result['winner']}  ({sa} 対 {sb})")
table.snap(CONFIG["final_hold_frames"])
table.save(CONFIG["out_mp4"])
p.disconnect()
print(f"動画: {CONFIG['out_mp4']}  フレーム数 {len(table.frames)}  長さ約 {len(table.frames)/CONFIG['fps']:.0f} 秒")

In [ ]:
# ===== 動画の表示と結果発表 =====
from base64 import b64encode
from IPython.display import HTML, display
mp4 = open(CONFIG["out_mp4"], "rb").read()
display(HTML(f'<video width=480 controls><source src="data:video/mp4;base64,{b64encode(mp4).decode()}" type="video/mp4"></video>'))
sa, sb = result["score"]
print("====== 結果発表 ======")
print(f"♥ロボットA の取得カード: {sa}枚  {[label('♠', n) for _, n in result['camp']['A']]}")
print(f"♠ロボットB の取得カード: {sb}枚  {[label('♥', n) for _, n in result['camp']['B']]}")
print(f"ドロー（やり直し）: {result['draws']}回")
print(f"勝者: {result['winner']}")